In [1]:
%pip install plotly -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Core Libraries ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display & Formatting ───────────────────────────────────────────────────────
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

## Part 1 : Query a Weather API 

In [3]:
# 1. Load data
df = pd.read_csv("NasaWeatherdata.csv", skiprows=15)


In [4]:
df.head(5)

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,CLOUD_AMT
0,2022,213,30.93,40.58,22.66,0.18,48.52,3.40,19.79
1,2022,214,30.01,38.79,21.79,0.06,49.87,3.24,22.09
2,2022,215,29.69,38.35,21.44,0.02,46.99,3.60,4.11
3,2022,216,30.06,38.95,21.37,0.00,48.71,2.95,21.89
4,2022,217,30.28,38.49,22.67,0.00,46.93,2.82,25.25


## Part 2 — Prepare the Time-Series Dataset 

In [5]:
def data_snapshot(df, name="Dataset"):
    print(f"{'='*60}\n📊  {name} Overview\n{'='*60}")
    print(f"Shape:        {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Duplicates:   {df.duplicated().sum():,}\n{'─'*60}")
    
    missing = df.isnull().sum()
    miss_df = pd.DataFrame({'Missing': missing, 'Missing %': (missing/len(df)*100).round(2), 'Dtype': df.dtypes})
    miss_df = miss_df[miss_df['Missing'] > 0].sort_values('Missing %', ascending=False)
    
    if len(miss_df) > 0:
        print("Missing Values:"); display(miss_df)
    else:
        print("✅ No missing values")
    
    print(f"{'─'*60}\nData Types:\n{df.dtypes.value_counts()}\n{'='*60}")


In [6]:
# 2. Health check
data_snapshot(df)


📊  Dataset Overview
Shape:        1,096 rows × 9 columns
Duplicates:   0
────────────────────────────────────────────────────────────
✅ No missing values
────────────────────────────────────────────────────────────
Data Types:
float64    7
int64      2
Name: count, dtype: int64


In [7]:
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,CLOUD_AMT
count,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000
mean,2023.581204,183.166971,22.584599,30.438659,15.769781,0.336177,52.500338,2.780356,25.270922
std,0.954094,105.510927,6.872355,7.621071,5.871208,2.368778,13.369643,0.805054,22.195351
min,2022.000000,1.000000,8.320000,13.750000,2.650000,0.000000,15.880000,0.750000,0.280000
25%,2023.000000,92.000000,15.957500,23.317500,10.410000,0.000000,43.107500,2.230000,8.432500
50%,2024.000000,183.000000,22.865000,31.030000,15.870000,0.000000,51.220000,2.780000,18.835000
75%,2024.000000,274.250000,29.332500,37.842500,21.327500,0.010000,62.592500,3.300000,34.730000
max,2025.000000,366.000000,35.050000,45.490000,27.510000,42.540000,84.600000,5.700000,99.890000


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   YEAR         1096 non-null   int64  
 1   DOY          1096 non-null   int64  
 2   T2M          1096 non-null   float64
 3   T2M_MAX      1096 non-null   float64
 4   T2M_MIN      1096 non-null   float64
 5   PRECTOTCORR  1096 non-null   float64
 6   RH2M         1096 non-null   float64
 7   WS2M         1096 non-null   float64
 8   CLOUD_AMT    1096 non-null   float64
dtypes: float64(7), int64(2)
memory usage: 77.2 KB


In [9]:
# check nulls 
df = df.replace(-999, pd.NA)
df.isna().sum()  

YEAR           0
DOY            0
T2M            0
T2M_MAX        0
T2M_MIN        0
PRECTOTCORR    0
RH2M           0
WS2M           0
CLOUD_AMT      0
dtype: int64

In [10]:
# check dublicates 
df.index.duplicated().sum()         

np.int64(0)

In [11]:
df['date'] = pd.to_datetime(df['YEAR'], format='%Y') + pd.to_timedelta(df['DOY'] - 1, unit='D')
df = df.set_index('date').drop(columns=['YEAR', 'DOY'])

In [12]:
df = df.rename(columns={
    'T2M': 'temp_avg_c',
    'T2M_MAX': 'temp_max_c',
    'T2M_MIN': 'temp_min_c',
    'PRECTOTCORR': 'precip_mm',
    'RH2M': 'humidity_pct',
    'WS2M': 'windspeed_ms',
    'CLOUD_AMT': 'cloud_cover_pct'
})

In [13]:
df.describe()

,temp_avg_c,temp_max_c,temp_min_c,precip_mm,humidity_pct,windspeed_ms,cloud_cover_pct
count,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000,1096.000000
mean,22.584599,30.438659,15.769781,0.336177,52.500338,2.780356,25.270922
std,6.872355,7.621071,5.871208,2.368778,13.369643,0.805054,22.195351
min,8.320000,13.750000,2.650000,0.000000,15.880000,0.750000,0.280000
25%,15.957500,23.317500,10.410000,0.000000,43.107500,2.230000,8.432500
50%,22.865000,31.030000,15.870000,0.000000,51.220000,2.780000,18.835000
75%,29.332500,37.842500,21.327500,0.010000,62.592500,3.300000,34.730000
max,35.050000,45.490000,27.510000,42.540000,84.600000,5.700000,99.890000


In [14]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1096 entries, 2022-08-01 to 2025-07-31
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   temp_avg_c       1096 non-null   float64
 1   temp_max_c       1096 non-null   float64
 2   temp_min_c       1096 non-null   float64
 3   precip_mm        1096 non-null   float64
 4   humidity_pct     1096 non-null   float64
 5   windspeed_ms     1096 non-null   float64
 6   cloud_cover_pct  1096 non-null   float64
dtypes: float64(7)
memory usage: 68.5 KB


In [15]:
df.index

DatetimeIndex(['2022-08-01', '2022-08-02', '2022-08-03', '2022-08-04',
               '2022-08-05', '2022-08-06', '2022-08-07', '2022-08-08',
               '2022-08-09', '2022-08-10',
               ...
               '2025-07-22', '2025-07-23', '2025-07-24', '2025-07-25',
               '2025-07-26', '2025-07-27', '2025-07-28', '2025-07-29',
               '2025-07-30', '2025-07-31'],
              dtype='datetime64[us]', name='date', length=1096, freq=None)

In [16]:
(df['precip_mm'] == 0).sum()

np.int64(795)

In [17]:
df.tail()

,temp_avg_c,temp_max_c,temp_min_c,precip_mm,humidity_pct,windspeed_ms,cloud_cover_pct
date,,,,,,,
2025-07-27,33.87,42.66,25.08,0.0,33.31,2.86,8.69
2025-07-28,32.88,41.39,24.69,0.0,34.17,2.94,17.90
2025-07-29,31.41,40.13,23.98,0.0,47.79,3.45,16.12
2025-07-30,30.00,38.03,22.83,0.0,45.85,3.01,12.74
2025-07-31,29.49,37.00,23.09,0.0,50.55,3.48,24.41


In [18]:
# Make sure of units 
import pandas as pd

NASA_POWER_UNITS = {
    'temp_avg_c': '°C',
    'temp_max_c': '°C',
    'temp_min_c': '°C',
    'precip_mm': 'mm/day',
    'humidity_pct': '%',
    'windspeed_ms': 'm/s',
    'cloud_cover_pct': '%',
}
def get_column_units(df):
    results = []
    for col in df.columns:
        unit = NASA_POWER_UNITS.get(col, '— (unknown)')
        results.append({
            'Column': col,
            'Unit': unit,
            'Dtype': str(df[col].dtype),
            'Non-Null': df[col].notna().sum(),
            'Null': df[col].isna().sum()
        })
    return pd.DataFrame(results)

In [19]:
get_column_units(df)

,Column,Unit,Dtype,Non-Null,Null
0,temp_avg_c,°C,float64,1096,0
1,temp_max_c,°C,float64,1096,0
2,temp_min_c,°C,float64,1096,0
3,precip_mm,mm/day,float64,1096,0
4,humidity_pct,%,float64,1096,0
5,windspeed_ms,m/s,float64,1096,0
6,cloud_cover_pct,%,float64,1096,0


In [20]:
df.to_csv("giza_weather_clean.csv")